In [1]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

# Visualization settings
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# ============================================================
# 2. LOAD DATA
# ============================================================

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

train = pd.read_csv(DATA_DIR / "train.csv",index_col=0)
test = pd.read_csv(DATA_DIR / "test.csv",index_col=0)

print("Training shape:", train.shape)
print("Testing shape:", test.shape)

Training shape: (103904, 24)
Testing shape: (25976, 24)


# <font color = "Orange"> Plan </font>

## Build 2 models:
### - Passenger and flight characteristics only model <font color = "Orange"> (model A) </font>: Age, Gender, Customer Type, Type of Travel, Class, Flight Distance, Departure and Arrival Delays
### - Full passenger experience model <font color = "Orange"> (model B) </font>: Everything in A and Online Boarding, Inflight Entertainment, Seat Comfort, Wifi, Other ratings
### - Neutral/Disatisfied -> 0, Satisfied -> 1

# <font color = "Orange">  1. Preparing data </font>

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

# Define target
target = "satisfaction"

# Separate features and target
X = train.drop(columns=[target, "id"])
y = train[target].map({
    "neutral or dissatisfied": 0,
    "satisfied": 1
})

# Remove identifier from test features too
X_test = test.drop(columns=["id"])

# Stratified train-validation split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Check distributions
print("Training:", y_train.value_counts(normalize=True))
print("Validation:", y_val.value_counts(normalize=True))
print("Kaggle test:", test[target].value_counts(normalize=True))

Training: satisfaction
0    0.566666
1    0.433334
Name: proportion, dtype: float64
Validation: satisfaction
0    0.566671
1    0.433329
Name: proportion, dtype: float64
Kaggle test: satisfaction
neutral or dissatisfied    0.561018
satisfied                  0.438982
Name: proportion, dtype: float64


# <font color = "Orange">  2. Building a Logistic Regression baseline </font>

### Logistic Regression cannot interpret strings, need to convert into numerical representations (via one-hot encoding), also need to standardize numerical features

## 2.1 Preprocessing

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression

# ============================================================
# DEFINE FEATURE GROUPS
# ============================================================

categorical_features = [
    "Gender",
    "Customer Type",
    "Type of Travel",
    "Class"
]

numerical_features = [
    "Age",
    "Flight Distance",
    "Departure Delay in Minutes",
    "Arrival Delay in Minutes"
]

# Wi-Fi is handled separately because 0 = Not Applicable
categorical_features_model_b = categorical_features + [
    "Inflight wifi service"
]

# ============================================================
# NUMERICAL PREPROCESSING
# ============================================================

numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    ("scaler", StandardScaler())
])

# ============================================================
# CATEGORICAL PREPROCESSING
# ============================================================

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# ============================================================
# MODEL A PREPROCESSING
# ============================================================

preprocessor_a = ColumnTransformer(transformers=[
    ("num", numerical_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

# ============================================================
# MODEL B PREPROCESSING
# ============================================================

preprocessor_b = ColumnTransformer(transformers=[
    ("num", numerical_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features_model_b),
    ("service", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), [
        "Online boarding",
        "Inflight entertainment",
        "Seat comfort",
        "On-board service",
        "Leg room service",
        "Cleanliness",
        "Checkin service",
        "Baggage handling",
        "Inflight service",
        "Food and drink",
        "Ease of Online booking",
        "Gate location",
        "Departure/Arrival time convenient"
    ])
])

## 2.2 Train and evaluate Logistic Regression baseline

In [ ]:
# Model A

model_a = Pipeline(steps=[
    ("preprocessor", preprocessor_a),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Model B
model_b = Pipeline(steps=[
    ("preprocessor", preprocessor_b),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Train models
model_a.fit(X_train, y_train)
model_b.fit(X_train, y_train)

print("Model A trained.")
print("Model B trained.")

Model A trained.
Model B trained.


## <font color = "Red">  Model A LR Baseline results </font>

In [8]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score
)
import matplotlib.pyplot as plt

y_pred = model_a.predict(X_val)
y_proba = model_a.predict_proba(X_val)[:, 1]

print("Accuracy:", accuracy_score(y_val, y_pred))
print("ROC-AUC:", roc_auc_score(y_val, y_proba))

print("\nClassification report:")
print(classification_report(y_val, y_pred))

print("\nConfusion matrix:")
print(confusion_matrix(y_val, y_pred))



Accuracy: 0.7863914152350705
ROC-AUC: 0.847368418322852

Classification report:
              precision    recall  f1-score   support

           0       0.82      0.80      0.81     11776
           1       0.74      0.77      0.76      9005

    accuracy                           0.79     20781
   macro avg       0.78      0.79      0.78     20781
weighted avg       0.79      0.79      0.79     20781


Confusion matrix:
[[9366 2410]
 [2029 6976]]


## <font color = "Red">  Model B LR Baseline results </font>

In [9]:
y_pred_b = model_b.predict(X_val)
y_proba_b = model_b.predict_proba(X_val)[:, 1]

print("Accuracy:", accuracy_score(y_val, y_pred_b))
print("ROC-AUC:", roc_auc_score(y_val, y_proba_b))

print("\nClassification report:")
print(classification_report(y_val, y_pred_b))

print("\nConfusion matrix:")
print(confusion_matrix(y_val, y_pred_b))

Accuracy: 0.9254607574226457
ROC-AUC: 0.9739874850626464

Classification report:
              precision    recall  f1-score   support

           0       0.93      0.94      0.93     11776
           1       0.92      0.91      0.91      9005

    accuracy                           0.93     20781
   macro avg       0.92      0.92      0.92     20781
weighted avg       0.93      0.93      0.93     20781


Confusion matrix:
[[11062   714]
 [  835  8170]]


### Substantial improvements in B, having more data on satisfaction ratings does matter, **but an airline doesn't necesarrily have those information before the flight even happens**

## <font color = "Red">  Side-by-side baseline comparison </font>

In [10]:
comparison = pd.DataFrame({
    "Model A": [
        accuracy_score(y_val, model_a.predict(X_val)),
        roc_auc_score(y_val, model_a.predict_proba(X_val)[:, 1])
    ],
    "Model B": [
        accuracy_score(y_val, model_b.predict(X_val)),
        roc_auc_score(y_val, model_b.predict_proba(X_val)[:, 1])
    ]
}, index=["Accuracy", "ROC-AUC"])

print(comparison)

           Model A   Model B
Accuracy  0.786391  0.925461
ROC-AUC   0.847368  0.973987


## 2.3 Investigating which feature are associated with satisfaction

In [11]:
# Extract fitted logistic regression classifier
classifier = model_b.named_steps["classifier"]

# Extract feature names after preprocessing
feature_names = model_b.named_steps["preprocessor"].get_feature_names_out()

# Pair feature names with coefficients
coefficients = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": classifier.coef_[0]
})

# Sort by coefficient
coefficients["Abs_Coefficient"] = coefficients["Coefficient"].abs()

coefficients = coefficients.sort_values(
    "Abs_Coefficient",
    ascending=False
)

print(coefficients.head(20))

                                       Feature  Coefficient  Abs_Coefficient
14                cat__Inflight wifi service_0     7.899604         7.899604
17                cat__Inflight wifi service_3    -3.649183         3.649183
19                cat__Inflight wifi service_5     3.387913         3.387913
16                cat__Inflight wifi service_2    -3.186401         3.186401
15                cat__Inflight wifi service_1    -2.787527         2.787527
10         cat__Type of Travel_Personal Travel    -1.807949         1.807949
9          cat__Type of Travel_Business travel     1.746277         1.746277
18                cat__Inflight wifi service_4    -1.726077         1.726077
8         cat__Customer Type_disloyal Customer    -1.455397         1.455397
7            cat__Customer Type_Loyal Customer     1.393725         1.393725
20                    service__Online boarding     1.018115         1.018115
11                         cat__Class_Business     0.608227         0.608227

### Consistent with Stage 1 EDA, but need to consider wifi rating 0 again. Its coefficient being so large means it was not dropped as the reference category. 

In [ ]:
wifi_encoder = (
    model_b.named_steps["preprocessor"]
    .named_transformers_["cat"]
    .named_steps["onehot"]
)

print(wifi_encoder.categories_)
print(wifi_encoder.drop_idx_)

[array(['Female', 'Male'], dtype=object), array(['Loyal Customer', 'disloyal Customer'], dtype=object), array(['Business travel', 'Personal Travel'], dtype=object), array(['Business', 'Eco', 'Eco Plus'], dtype=object), array([0, 1, 2, 3, 4, 5], dtype=object)]
None


##   2.3 Revised baseline models 

In [16]:
# Define target
target = "satisfaction"

# Separate features and target
X = train.drop(columns=[target, "id"])

y = train[target].map({
    "neutral or dissatisfied": 0,
    "satisfied": 1
})

# Prepare test features
X_test = test.drop(columns=["id"])


# Separate Wi-Fi applicability and rating
def prepare_wifi_features(df):
    df = df.copy()

    df["wifi_rating_applicable"] = (
        df["Inflight wifi service"] != 0
    ).astype(int)

    df["wifi_rating"] = (
        df["Inflight wifi service"].replace(0, np.nan)
    )

    df = df.drop(columns=["Inflight wifi service"])

    return df


X = prepare_wifi_features(X)
X_test = prepare_wifi_features(X_test)

print("Training features:", X.shape)
print("Test features:", X_test.shape)

print("\nWi-Fi applicability:")
print(X["wifi_rating_applicable"].value_counts())

print("\nWi-Fi rating:")
print(X["wifi_rating"].describe())

print("\nMissing Wi-Fi ratings:")
print(X["wifi_rating"].isna().sum())

Training features: (103904, 23)
Test features: (25976, 24)

Wi-Fi applicability:
wifi_rating_applicable
1    100801
0      3103
Name: count, dtype: int64

Wi-Fi rating:
count    100801.000000
mean          2.813712
std           1.257367
min           1.000000
25%           2.000000
50%           3.000000
75%           4.000000
max           5.000000
Name: wifi_rating, dtype: float64

Missing Wi-Fi ratings:
3103


In [20]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)

Training set: (83123, 23)
Validation set: (20781, 23)


## <font color = "Red">  Rebuilt Model B LR Baseline results </font>

In [21]:
categorical_features = [
    "Gender",
    "Customer Type",
    "Type of Travel",
    "Class",
    "wifi_rating_applicable"
]

numerical_features = [
    "Age",
    "Flight Distance",
    "Departure Delay in Minutes",
    "Arrival Delay in Minutes",
    "wifi_rating"
]

service_features = [
    "Departure/Arrival time convenient",
    "Ease of Online booking",
    "Gate location",
    "Food and drink",
    "Online boarding",
    "Seat comfort",
    "Inflight entertainment",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Inflight service",
    "Cleanliness"
]

# Numerical features
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=False
    )),
    ("scaler", StandardScaler())
])

# Categorical features
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        drop="first",
        handle_unknown="ignore"
    ))
])

# Service rating features
service_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


# Combine transformations
preprocessor_b = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features),
        ("service", service_transformer, service_features)
    ]
)

model_b = Pipeline(steps=[
    ("preprocessor", preprocessor_b),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model_b.fit(X_train, y_train)

# Predictions
y_pred_b = model_b.predict(X_val)

# Predicted probability of satisfaction
y_prob_b = model_b.predict_proba(X_val)[:, 1]


# Evaluation metrics
print("Model B — Logistic Regression")

print("\nAccuracy:")
print(accuracy_score(y_val, y_pred_b))

print("\nROC-AUC:")
print(roc_auc_score(y_val, y_prob_b))

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_pred_b,
    target_names=["Neutral/Dissatisfied", "Satisfied"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred_b))

Model B — Logistic Regression

Accuracy:
0.9017371637553534

ROC-AUC:
0.9646426803949497

Classification Report:
                      precision    recall  f1-score   support

Neutral/Dissatisfied       0.92      0.91      0.91     11776
           Satisfied       0.88      0.89      0.89      9005

            accuracy                           0.90     20781
           macro avg       0.90      0.90      0.90     20781
        weighted avg       0.90      0.90      0.90     20781


Confusion Matrix:
[[10717  1059]
 [  983  8022]]


### Accuracy and AUC decreased

In [22]:
cat_pipe = (
    model_b.named_steps["preprocessor"]
    .named_transformers_["cat"]
)

encoder = cat_pipe.named_steps["onehot"]

for feature, categories, drop_idx in zip(
    categorical_features,
    encoder.categories_,
    encoder.drop_idx_
):
    reference = (
        categories[drop_idx]
        if drop_idx is not None
        else None
    )

    print(f"\nFeature: {feature}")
    print("All categories:", categories)
    print("Reference category:", reference)
    
# Get transformed feature names
feature_names = (
    model_b.named_steps["preprocessor"]
    .get_feature_names_out()
)

# Get model coefficients
coefficients = (
    model_b.named_steps["classifier"]
    .coef_[0]
)

# Create coefficient dataframe
coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

coef_df["Abs_Coefficient"] = (
    coef_df["Coefficient"].abs()
)

# Sort by absolute coefficient
coef_df = coef_df.sort_values(
    by="Abs_Coefficient",
    ascending=False
)

print(coef_df.to_string(index=False))


Feature: Gender
All categories: ['Female' 'Male']
Reference category: Female

Feature: Customer Type
All categories: ['Loyal Customer' 'disloyal Customer']
Reference category: Loyal Customer

Feature: Type of Travel
All categories: ['Business travel' 'Personal Travel']
Reference category: Business travel

Feature: Class
All categories: ['Business' 'Eco' 'Eco Plus']
Reference category: Business

Feature: wifi_rating_applicable
All categories: [0 1]
Reference category: 0
                                   Feature  Coefficient  Abs_Coefficient
             cat__wifi_rating_applicable_1   -10.832860        10.832860
       cat__Type of Travel_Personal Travel    -3.244598         3.244598
      cat__Customer Type_disloyal Customer    -2.761278         2.761278
                       cat__Class_Eco Plus    -1.180889         1.180889
                  service__Online boarding     1.144226         1.144226
                          num__wifi_rating     1.053059         1.053059
              

### Now wifi rating 1 jumped to -10.8

## <font color = "Red">  Rebuilt Model A LR Baseline results </font>

In [23]:
# Model A: Passenger and flight characteristics

categorical_features_a = [
    "Gender",
    "Customer Type",
    "Type of Travel",
    "Class",
    "wifi_rating_applicable"
]

numerical_features_a = [
    "Age",
    "Flight Distance",
    "Departure Delay in Minutes",
    "Arrival Delay in Minutes",
    "wifi_rating"
]

# Numerical features
numerical_transformer_a = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    ("scaler", StandardScaler())
])

# Wi-Fi rating
wifi_transformer_a = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical features
categorical_transformer_a = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        drop="first",
        handle_unknown="ignore"
    ))
])

# Combine preprocessing
preprocessor_a = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer_a, numerical_features_a),
        ("wifi", wifi_transformer_a, ["wifi_rating"]),
        ("cat", categorical_transformer_a, categorical_features_a)
    ]
)

model_a = Pipeline(steps=[
    ("preprocessor", preprocessor_a),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model_a.fit(X_train, y_train)

# Predictions
y_pred_a = model_a.predict(X_val)
y_prob_a = model_a.predict_proba(X_val)[:, 1]


# Evaluation
print("Model A — Logistic Regression")

print("\nAccuracy:")
print(accuracy_score(y_val, y_pred_a))

print("\nROC-AUC:")
print(roc_auc_score(y_val, y_prob_a))

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_pred_a,
    target_names=["Neutral/Dissatisfied", "Satisfied"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred_a))

Model A — Logistic Regression

Accuracy:
0.815937635339974

ROC-AUC:
0.9163601978746712

Classification Report:
                      precision    recall  f1-score   support

Neutral/Dissatisfied       0.83      0.85      0.84     11776
           Satisfied       0.80      0.77      0.78      9005

            accuracy                           0.82     20781
           macro avg       0.81      0.81      0.81     20781
        weighted avg       0.82      0.82      0.82     20781


Confusion Matrix:
[[10011  1765]
 [ 2060  6945]]


## <font color = "Red">  Side-by-side rebuilt baseline comparison </font>

In [24]:
comparison = pd.DataFrame({
    "Model A": [
        accuracy_score(y_val, model_a.predict(X_val)),
        roc_auc_score(y_val, model_a.predict_proba(X_val)[:, 1])
    ],
    "Model B": [
        accuracy_score(y_val, model_b.predict(X_val)),
        roc_auc_score(y_val, model_b.predict_proba(X_val)[:, 1])
    ]
}, index=["Accuracy", "ROC-AUC"])

print(comparison)

           Model A   Model B
Accuracy  0.815938  0.901737
ROC-AUC   0.916360  0.964643


## Passenger and flight characteristics already provide strong predictive information, but service ratings substantially improve the model's ability to classify customer satisfaction.

# <font color = "Orange">  3. Random Forest </font>

In [25]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Feature groups
categorical_features_rf = [
    "Gender",
    "Customer Type",
    "Type of Travel",
    "Class",
    "wifi_rating_applicable"
]

numerical_features_rf = [
    "Age",
    "Flight Distance",
    "Departure Delay in Minutes",
    "Arrival Delay in Minutes"
]

wifi_features_rf = ["wifi_rating"]

service_features_rf = [
    "Departure/Arrival time convenient",
    "Ease of Online booking",
    "Gate location",
    "Food and drink",
    "Online boarding",
    "Seat comfort",
    "Inflight entertainment",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Inflight service",
    "Cleanliness"
]

# Numerical features: median imputation + missingness indicators
numerical_transformer_rf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True))
])

# Wi-Fi rating: median imputation, without an additional missingness indicator
wifi_transformer_rf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical features: imputation + one-hot encoding
categorical_transformer_rf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Service ratings: median imputation
service_transformer_rf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Model A: passenger and flight characteristics
preprocessor_rf_a = ColumnTransformer(transformers=[
    ("num", numerical_transformer_rf, numerical_features_rf),
    ("wifi", wifi_transformer_rf, wifi_features_rf),
    ("cat", categorical_transformer_rf, categorical_features_rf)
])

# Model B: Model A + service ratings
preprocessor_rf_b = ColumnTransformer(transformers=[
    ("num", numerical_transformer_rf, numerical_features_rf),
    ("wifi", wifi_transformer_rf, wifi_features_rf),
    ("cat", categorical_transformer_rf, categorical_features_rf),
    ("service", service_transformer_rf, service_features)
])

## 3.1 Train both models

In [26]:
# Model A
model_rf_a = Pipeline(steps=[
    ("preprocessor", preprocessor_rf_a),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ))
])

model_rf_a.fit(X_train, y_train)


# Model B
model_rf_b = Pipeline(steps=[
    ("preprocessor", preprocessor_rf_b),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ))
])

model_rf_b.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](23,)","['Gender','Customer Type','Age',...,'Arrival Delay in Minutes', 'wifi_rating_applicable','wifi_rating']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,23
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('wifi', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By s

## <font color = "Red">  Evaluate model A and model B RF </font>

In [27]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score
)

# Predictions
y_pred_rf_a = model_rf_a.predict(X_val)
y_prob_rf_a = model_rf_a.predict_proba(X_val)[:, 1]

y_pred_rf_b = model_rf_b.predict(X_val)
y_prob_rf_b = model_rf_b.predict_proba(X_val)[:, 1]


# Evaluation function
def evaluate_model(name, y_true, y_pred, y_prob):
    print(f"\n{name}")
    print("-" * 40)

    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"ROC-AUC:  {roc_auc_score(y_true, y_prob):.4f}")
    print(f"Macro F1: {f1_score(y_true, y_pred, average='macro'):.4f}")

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))


evaluate_model("Random Forest - Model A", y_val, y_pred_rf_a, y_prob_rf_a)
evaluate_model("Random Forest - Model B", y_val, y_pred_rf_b, y_prob_rf_b)


Random Forest - Model A
----------------------------------------
Accuracy: 0.8604
ROC-AUC:  0.9391
Macro F1: 0.8587

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.86      0.87     11776
           1       0.82      0.87      0.84      9005

    accuracy                           0.86     20781
   macro avg       0.86      0.86      0.86     20781
weighted avg       0.86      0.86      0.86     20781

Confusion Matrix:
[[10070  1706]
 [ 1195  7810]]

Random Forest - Model B
----------------------------------------
Accuracy: 0.9628
ROC-AUC:  0.9945
Macro F1: 0.9620

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.98      0.97     11776
           1       0.97      0.95      0.96      9005

    accuracy                           0.96     20781
   macro avg       0.96      0.96      0.96     20781
weighted avg       0.96      0.96      0.96     20781

Confusion Matri

## Random forest outperforms LR in both, model B oddly high AUC, need sanity check

In [28]:
def compare_train_validation(name, model, X_train, y_train, X_val, y_val):
    # Training predictions
    train_pred = model.predict(X_train)
    train_prob = model.predict_proba(X_train)[:, 1]

    # Validation predictions
    val_pred = model.predict(X_val)
    val_prob = model.predict_proba(X_val)[:, 1]

    print(f"\n{name}")
    print("-" * 45)

    print("Training:")
    print(f"Accuracy: {accuracy_score(y_train, train_pred):.4f}")
    print(f"ROC-AUC:  {roc_auc_score(y_train, train_prob):.4f}")
    print(f"Macro F1: {f1_score(y_train, train_pred, average='macro'):.4f}")

    print("\nValidation:")
    print(f"Accuracy: {accuracy_score(y_val, val_pred):.4f}")
    print(f"ROC-AUC:  {roc_auc_score(y_val, val_prob):.4f}")
    print(f"Macro F1: {f1_score(y_val, val_pred, average='macro'):.4f}")


compare_train_validation(
    "Random Forest - Model A",
    model_rf_a,
    X_train, y_train,
    X_val, y_val
)

compare_train_validation(
    "Random Forest - Model B",
    model_rf_b,
    X_train, y_train,
    X_val, y_val
)


Random Forest - Model A
---------------------------------------------
Training:
Accuracy: 0.9993
ROC-AUC:  1.0000
Macro F1: 0.9993

Validation:
Accuracy: 0.8604
ROC-AUC:  0.9391
Macro F1: 0.8587

Random Forest - Model B
---------------------------------------------
Training:
Accuracy: 1.0000
ROC-AUC:  1.0000
Macro F1: 1.0000

Validation:
Accuracy: 0.9628
ROC-AUC:  0.9945
Macro F1: 0.9620


## <font color = "Red"> Both models are overfitting, need constraints on Forest </font>

In [30]:
# Constrained Random Forest - Model A
model_rf_a_regularized = Pipeline(steps=[
    ("preprocessor", preprocessor_rf_a),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_features="sqrt",
        max_depth=20,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ))
])

model_rf_a_regularized.fit(X_train, y_train)


# Constrained Random Forest - Model B
model_rf_b_regularized = Pipeline(steps=[
    ("preprocessor", preprocessor_rf_b),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_features="sqrt",
        max_depth=20,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ))
])

model_rf_b_regularized.fit(X_train, y_train)

y_pred_rf_a_reg = model_rf_a_regularized.predict(X_val)
y_prob_rf_a_reg = model_rf_a_regularized.predict_proba(X_val)[:, 1]

y_pred_rf_b_reg = model_rf_b_regularized.predict(X_val)
y_prob_rf_b_reg = model_rf_b_regularized.predict_proba(X_val)[:, 1]

evaluate_model(
    "Regularized Random Forest - Model A",
    y_val, y_pred_rf_a_reg, y_prob_rf_a_reg
)

evaluate_model(
    "Regularized Random Forest - Model B",
    y_val, y_pred_rf_b_reg, y_prob_rf_b_reg
)

compare_train_validation(
    "Random Forest - Model A",
    model_rf_a_regularized,
    X_train, y_train,
    X_val, y_val
)

compare_train_validation(
    "Random Forest - Model B",
    model_rf_b_regularized,
    X_train, y_train,
    X_val, y_val
)


Regularized Random Forest - Model A
----------------------------------------
Accuracy: 0.8788
ROC-AUC:  0.9497
Macro F1: 0.8780

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.85      0.89     11776
           1       0.82      0.92      0.87      9005

    accuracy                           0.88     20781
   macro avg       0.88      0.88      0.88     20781
weighted avg       0.88      0.88      0.88     20781

Confusion Matrix:
[[10004  1772]
 [  746  8259]]

Regularized Random Forest - Model B
----------------------------------------
Accuracy: 0.9605
ROC-AUC:  0.9939
Macro F1: 0.9597

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.97      0.97     11776
           1       0.96      0.94      0.95      9005

    accuracy                           0.96     20781
   macro avg       0.96      0.96      0.96     20781
weighted avg       0.96      0.96      0.96   

## Much better (and reliable) performance

## 3.2 Check Feature Importance 

## Check Feature Importance in B 

In [31]:
from sklearn.inspection import permutation_importance

permutation_result = permutation_importance(
    model_rf_b_regularized,
    X_val,
    y_val,
    n_repeats=5,
    random_state=42,
    scoring="f1_macro",
    n_jobs=-1
)

permutation_importance_df = pd.DataFrame({
    "Feature": X_val.columns,
    "Importance": permutation_result.importances_mean,
    "Std": permutation_result.importances_std
})

permutation_importance_df = permutation_importance_df.sort_values(
    by="Importance",
    ascending=False
)

permutation_importance_df

,Feature,Importance,Std
3,Type of Travel,0.149947,0.001523
22,wifi_rating,0.091808,0.001357
1,Customer Type,0.063590,0.000804
21,wifi_rating_applicable,0.032905,0.000259
10,Online boarding,0.030352,0.001212
16,Checkin service,0.022924,0.000398
4,Class,0.018754,0.000953
15,Baggage handling,0.017520,0.000335
11,Seat comfort,0.012052,0.000313
17,Inflight service,0.011726,0.000815


## Check Feature Importance in A and interpret

In [32]:
permutation_result_a = permutation_importance(
    model_rf_a_regularized,
    X_val,
    y_val,
    n_repeats=5,
    random_state=42,
    scoring="f1_macro",
    n_jobs=-1
)

permutation_importance_df_a = pd.DataFrame({
    "Feature": X_val.columns,
    "Importance": permutation_result_a.importances_mean,
    "Std": permutation_result_a.importances_std
})

permutation_importance_df_a = (
    permutation_importance_df_a
    .sort_values(by="Importance", ascending=False)
)

permutation_importance_df_a

,Feature,Importance,Std
22,wifi_rating,0.122520,0.002650
4,Class,0.082998,0.002152
3,Type of Travel,0.079392,0.001778
1,Customer Type,0.054350,0.000811
21,wifi_rating_applicable,0.029826,0.000379
2,Age,0.022395,0.000481
20,Arrival Delay in Minutes,0.003693,0.000712
5,Flight Distance,0.001934,0.000680
6,Departure/Arrival time convenient,0.000000,0.000000
14,Leg room service,0.000000,0.000000


## <font color = "Aquamarine"> Feature interpretation </font>

## <font color = "Red"> In B: </font>:
### - Type of Travel most important
### - Wifi suprisingly (or not) important
### - Customer Type more important than service ratings

## <font color = "Red"> In A: </font>:
### - Relies heavily on passenger and travel charateristics

### But this doesnt mean prioritizing wifi improvement (although that would be nice)!!

## <font color = "Red">  Both: </font>:
### - Service ratings provide information that overlaps with some characterstics, because Type of Travel becomes substantially more important and Class and Age become less important in B compared to A
### - Wifi still stays important

# <font color = "Orange">  4. Final evaluations with test set </font>

In [33]:
y_test = test["satisfaction"].map({
    "neutral or dissatisfied": 0,
    "satisfied": 1
})

# Logistic Regression
y_pred_lr_a_test = model_a.predict(X_test)
y_prob_lr_a_test = model_a.predict_proba(X_test)[:, 1]

y_pred_lr_b_test = model_b.predict(X_test)
y_prob_lr_b_test = model_b.predict_proba(X_test)[:, 1]


# Regularized Random Forest
y_pred_rf_a_test = model_rf_a_regularized.predict(X_test)
y_prob_rf_a_test = model_rf_a_regularized.predict_proba(X_test)[:, 1]

y_pred_rf_b_test = model_rf_b_regularized.predict(X_test)
y_prob_rf_b_test = model_rf_b_regularized.predict_proba(X_test)[:, 1]

evaluate_model("Logistic Regression - Model A (Test)",
               y_test, y_pred_lr_a_test, y_prob_lr_a_test)

evaluate_model("Logistic Regression - Model B (Test)",
               y_test, y_pred_lr_b_test, y_prob_lr_b_test)

evaluate_model("Regularized Random Forest - Model A (Test)",
               y_test, y_pred_rf_a_test, y_prob_rf_a_test)

evaluate_model("Regularized Random Forest - Model B (Test)",
               y_test, y_pred_rf_b_test, y_prob_rf_b_test)


Logistic Regression - Model A (Test)
----------------------------------------
Accuracy: 0.8089
ROC-AUC:  0.9120
Macro F1: 0.8050

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.85      0.83     14573
           1       0.80      0.76      0.78     11403

    accuracy                           0.81     25976
   macro avg       0.81      0.80      0.80     25976
weighted avg       0.81      0.81      0.81     25976

Confusion Matrix:
[[12343  2230]
 [ 2735  8668]]

Logistic Regression - Model B (Test)
----------------------------------------
Accuracy: 0.9014
ROC-AUC:  0.9638
Macro F1: 0.8999

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.91      0.91     14573
           1       0.89      0.89      0.89     11403

    accuracy                           0.90     25976
   macro avg       0.90      0.90      0.90     25976
weighted avg       0.90      0.90      0.90 

## <font color = "Orange">  Good results (Yipee!!!!) </font>

### <font color = "Red"> Regularized Random Forest Model B </font> stays the strongest candidate, <font color = "Red"> Regularized Random Forest  Model A </font> not too bad either in second